# Прављење апликација за генерисање слика (OpenAI)

Више од генерисања текста нуде LLM модели. Можете такође генерисати слике из текстуалних описа. Слике као модалитет су корисне у МедТеку, архитектури, туризму, развоју игара, маркетингу и још много тога. У овој лекцији радимо са данашњим **GPT Image** моделима на OpenAI платформи.

## Циљеви учења

До краја ове лекције моћи ћете да:

- Објасните шта је генерисање слика и где је корисно.
- Разумете породицу модела `gpt-image` и како се разликује од старих DALL·E модела.
- Направите апликацију за генерисање слика и уређујете слике.

## Шта је генерисање слика?

Модели за генерисање слика креирају слике на основу текстуалне поруке. Савремени модели као што је `gpt-image` уче везу између текста и слика током тренинга, а затим постепено претварају насумични шум у слику која одговара вашем опису.

Две познате породице модела слика су:

- **`gpt-image` (OpenAI)** — тренутна генерација која се користи у овој лекцији. Подржава генерисање слика из текста и уређивање слика (инапеинтинг са маском).
- **Midjourney** — популаран модел треће стране са сопственом услугом и радним током заснованим на Discord-у.

> Старији OpenAI модели слика — **DALL·E 2** и **DALL·E 3** — су наслеђени. DALL·E 3 више није доступан за нове имплементације, а функције као што је `create_variation` постојале су само у DALL·E 2. За нове апликације користите `gpt-image` моделе.

> **Важно:** `gpt-image` модели враћају генерисану слику као **base64** (`b64_json`), а не као URL. Ваш код декодује base64 стринг у бајтове и чува га — нема URL слике за преузимање.


## Прављење ваше прве апликације за генерисање слика

Шта је потребно за прављење апликације за генерисање слика? Требају вам следеће библиотеке:

- **python-dotenv**, ову библиотеку вам топло препоручујемо да користите како бисте чували своје тајне у *.env* фајлу, ван кода.
- **openai**, ова библиотека вам треба за интеракцију са OpenAI API-јем.
- **pillow**, за рад са сликама у Пајтону.


1. Направите фајл *.env* са следећим садржајем:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Прикупите горе наведене библиотеке у датотеку под називом *requirements.txt* као што следи:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Затим, креирајте виртуелно окружење и инсталирајте библиотеке:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> За Windows, користите следеће команде за креирање и активирање вашег виртуелног окружења:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Додајте следећи код у фајл под називом *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Хајде да објаснимо овај код:

- Прво, увозимо библиотеке које нам требају, укључујући OpenAI библиотеку, dotenv библиотеку, base64 модул и Pillow библиотеку.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Након тога, креирамо клијента, који чита API кључ из вашег ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Затим креирамо слику:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` модели враћају слику као **base64** стринг у `data[0].b64_json`. Ми га декодујемо у бајтове и пишемо у фајл — нема URL за преузимање.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- На крају, отварамо слику и користимо стандардни прегледач слика за приказ:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Више детаља о креирању слике

Погледајмо параметре `images.generate`:

- **model**, је модел слике који се користи (на пример `gpt-image-2`).
- **prompt**, је текстуални опис који се користи за креирање слике. Овде је "Зека на коњу, држи лижућку, на магловитој ливади где расту нарциси".
- **size**, је величина генерисане слике (на пример `1024x1024`, `1536x1024`, `1024x1536`, или `"auto"`).
- **n**, је број генерисаних слика. Овде генеришемо једну.

> Модели слика не узимају `temperature` параметар — то је параметар за контролу генерисања текста. Да бисте добили разноликост, позовите API поново; да бисте смањили разноликост, учините ваш опис прецизнијим.

## Додатне могућности генерисања слика

Видели сте како се генерише слика са неколико линија Питхон кода. `gpt-image` модели могу такође **уредити** постојећу слику. Пружањем слике, опционалне **маске** (која означава област за промену), и описа, можете изменити део слике — на пример, додати шешир нашем зецу.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Основна слика садржи само зеца; финална слика има шешир на зецу.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
